In [33]:
import pandas as pd
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split

# 1. Cargar el dataset desde la ruta especificada
df = pd.read_csv('datasets/dataset_fraude_sintetico.csv')


X = df.drop(columns=['es_fraude'])
y = df['es_fraude']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f"Dataset cargado con éxito. Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

print(df['es_fraude'].value_counts())

Dataset cargado con éxito. Filas: 85000, Columnas: 6
es_fraude
0    80000
1     5000
Name: count, dtype: int64


In [34]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

print("TensorFlow Versión:", tf.__version__)

# 1. 🌟 PASO CRÍTICO: Escalar los datos (Estandarización)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Arquitectura de la Red Neuronal
modelo_dl = Sequential([
    # Capa de Entrada y Primera Capa Oculta (32 neuronas)
    Dense(6, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    # Dropout apaga el 30% de las neuronas al azar para evitar que memorice (Overfitting)
    Dropout(0.3), 
    
    # Segunda Capa Oculta (16 neuronas)
    Dense(16, activation='relu'),
    Dropout(0.2),
    
    # Capa de Salida (1 sola neurona con activación Sigmoid para dar un % de probabilidad)
    Dense(1, activation='sigmoid')
])

# 3. Compilación del Modelo (Definir cómo va a aprender)
modelo_dl.compile(
    optimizer='adam', # El motor de aprendizaje más eficiente
    loss='binary_crossentropy', # La función matemática para errores binarios (Fraude o No)
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 4. Entrenamiento (Epochs = cuántas veces verá el dataset completo)
print("🚀 Entrenando la Red Neuronal...")
historial = modelo_dl.fit(
    X_train_scaled, 
    y_train,
    epochs=3,          # Daremos 50 pasadas de entrenamiento
    batch_size=32,      # Aprenderá en bloques de 32 registros a la vez
    validation_split=0.2, # Usará un 20% del X_train_scaled como evaluador en vivo
    verbose=1           # Ponlo en 0 si no quieres ver la barra de progreso
)

# 5. Predicciones y Evaluación
print("\n📊 Evaluando el modelo en el set de prueba...")
# En DL, predict te da la probabilidad cruda (ej. 0.85)
y_pred_proba_dl = modelo_dl.predict(X_test_scaled)

# Convertimos probabilidades mayores al 50% en clase 1 (Fraude)
y_pred_dl = (y_pred_proba_dl > 0.5).astype(int)

# 6. Mostrar Resultados
roc_auc_dl = roc_auc_score(y_test, y_pred_dl)
print(f"\nROC-AUC Score (Deep Learning): {roc_auc_dl:.4f}")
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred_dl))

TensorFlow Versión: 2.16.2
🚀 Entrenando la Red Neuronal...
Epoch 1/3


/Users/alejandropulido/Documents/software-projects/IA/IA Training/biometric_fraud/env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1700/1700 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9796 - auc: 0.9521 - loss: 0.0989 - val_accuracy: 0.9985 - val_auc: 1.0000 - val_loss: 0.0061
Epoch 2/3
1700/1700 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9929 - auc: 0.9963 - loss: 0.0263 - val_accuracy: 0.9991 - val_auc: 1.0000 - val_loss: 0.0032
Epoch 3/3
1700/1700 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9946 - auc: 0.9987 - loss: 0.0172 - val_accuracy: 0.9991 - val_auc: 1.0000 - val_loss: 0.0018

📊 Evaluando el modelo en el set de prueba...
532/532 ━━━━━━━━━━━━━━━━━━━━ 0s 677us/step

ROC-AUC Score (Deep Learning): 0.9970

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     16000
           1       1.00      0.99      1.00      1000

    accuracy                           1.00     17000
   macro avg       1.00      1.00      1.00     17000
weighted avg       1.00      1.00      1.00     17000



In [35]:
import joblib

# 1. Guardar el modelo de Red Neuronal (Formato nativo de Keras)
modelo_dl.save('modelo_dl_fraudes.keras')

# 2. Guardar el Escalar (¡CRÍTICO para que la red entienda los datos en vivo!)
joblib.dump(scaler, 'scaler_dl.joblib')

# 3. Guardar el orden exacto de las columnas de entrenamiento
columnas_entrenamiento = list(X_train.columns)
joblib.dump(columnas_entrenamiento, 'columnas_dl.joblib')

print("✅ Artefactos de Deep Learning exportados con éxito.")

✅ Artefactos de Deep Learning exportados con éxito.
